# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedasker1/FlyRank_Repo/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

*The playbook outputs a ranked queue prioritizing pages with the highest risk of traffic decay.
Actions: Immediate_Refresh_Review, Monitor_CTR, or Low_Priority.
Reason Codes:  High_Decay_Risk: Model probability > 0.65.Volume_CTR_Opportunity: High impressions but dropping CTR, requiring metadata review.Stable_Traffic: Model probability < 0.30.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import json

# 1. Environment Setup
IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR], check=True)
    if os.path.basename(os.getcwd()) != REPO_DIR:
        os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

# 2. Load Data and Simulate Model Output
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df['is_declining'] = (df['trend_direction'].str.lower() == 'down').astype(int)

features = ['content_age_days', 'impressions_90d', 'ctr', 'avg_position', 'word_count']
df_clean = df.dropna(subset=features + ['client_id', 'is_declining', 'content_id']).copy()

# Fast retrain for scoring
rf_model = RandomForestClassifier(n_estimators=50, max_depth=5, class_weight='balanced', random_state=42)
rf_model.fit(df_clean[features], df_clean['is_declining'])
df_clean['decay_probability'] = rf_model.predict_proba(df_clean[features])[:, 1]

# 3. Assign Actions and Reason Codes
conditions = [
    (df_clean['decay_probability'] > 0.65),
    (df_clean['impressions_90d'] > 1000) & (df_clean['ctr'] < 0.05),
]
actions = ['Immediate_Refresh_Review', 'Monitor_CTR']
reasons = ['High_Decay_Risk', 'Volume_CTR_Opportunity']

df_clean['action'] = np.select(conditions, actions, default='Low_Priority')
df_clean['reason_code'] = np.select(conditions, reasons, default='Stable_Traffic')

# 4. Show the top of the queue
actionable_queue = df_clean[df_clean['action'] != 'Low_Priority'].sort_values('decay_probability', ascending=False)
print(f"Total actionable pages: {len(actionable_queue)}")
actionable_queue[['content_id', 'decay_probability', 'action', 'reason_code', 'impressions_90d']].head(5)

Total actionable pages: 2900


,content_id,decay_probability,action,reason_code,impressions_90d
18559,content_0d9c0ed65840,0.733785,Immediate_Refresh_Review,High_Decay_Risk,382
28121,content_ffeed86ef360,0.718231,Immediate_Refresh_Review,High_Decay_Risk,383
17547,content_f552433bab3c,0.717541,Immediate_Refresh_Review,High_Decay_Risk,337
24240,content_bbe9c7753ee0,0.715834,Immediate_Refresh_Review,High_Decay_Risk,16090
6842,content_91fefd1726b3,0.713548,Immediate_Refresh_Review,High_Decay_Risk,2748


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

*Intended Use: This playbook is a decision-support tool designed to help content strategists prioritize their weekly review queue. It highlights where human attention is most likely to yield ROI.
Limits: This is an observational ranking, not a causal engine. It cannot guarantee that rewriting a page will recover its traffic. It is also blind to off-page factors (like backlink loss) or broad algorithmic penalties.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Limits mapped: Observational signals only. Off-page SEO factors are out of scope.")

Limits mapped: Observational signals only. Off-page SEO factors are out of scope.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

*Human Review Rules: Before acting on a recommendation, a reviewer must verify that the traffic drop isn't due to seasonal trends (e.g., Christmas content dropping in January) or page consolidation (traffic shifted to a sibling URL).
The No-Go List (Never Automate):  Never automatically rewrite titles or meta descriptions based on this score alone.Never automatically unpublish or redirect (prune) a page without human verification. *

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("No-Go List Active: Automation of content pruning or metadata overwriting is strictly blocked.")

No-Go List Active: Automation of content pruning or metadata overwriting is strictly blocked.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

*Monitoring: We will monitor the Precision@50 metric on a monthly basis.
Retrain Triggers: The model should be retrained if Precision@50 drops below the rule-based baseline (e.g., 0.240), or automatically every 6 months to adapt to changes in Google's SERP layouts and user CTR behaviors*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def check_retrain_trigger(current_precision, baseline_precision=0.240):
    if current_precision < baseline_precision:
        return "TRIGGER RETRAIN: Model is underperforming the naive baseline."
    return "STATUS OK: Model is performing within acceptable limits."

print(check_retrain_trigger(0.20)) # Example of a failing trigger

TRIGGER RETRAIN: Model is underperforming the naive baseline.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

*Exporting the prioritized queue and the run's metadata to work/outputs/ so the research paper can ingest them safely. These raw CSVs will stay out of git.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create outputs directory
os.makedirs('work/outputs', exist_ok=True)

# Export the queue
csv_path = 'work/outputs/playbook_action_queue.csv'
actionable_queue.to_csv(csv_path, index=False)

# Export the receipts (metrics)
metrics = {
    "total_pages_scored": len(df_clean),
    "actionable_pages": len(actionable_queue),
    "high_risk_count": int((df_clean['reason_code'] == 'High_Decay_Risk').sum()),
    "precision_at_50": 0.54 # Placeholder from previous evaluation
}

json_path = 'work/outputs/playbook_metrics.json'
with open(json_path, 'w') as f:
    json.dump(metrics, f, indent=4)

print(f"Successfully exported queue to {csv_path}")
print(f"Successfully exported metrics to {json_path}")

Successfully exported queue to work/outputs/playbook_action_queue.csv
Successfully exported metrics to work/outputs/playbook_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.